# Name That LLM — CPU submission

1. Attach the **Name That LLM competition** and your **private model-bundle dataset**.
2. Set **Accelerator: None** and **Internet: Off**.
3. Run all cells, then save a **Save & Run All** version.
4. Submit the successful saved notebook version to the competition.

No training or online downloads are needed. The bundle contains our trusted model
and Python 3.12/Linux wheels. It may be attached extracted or as the original ZIP.
This notebook predicts every row Kaggle supplies, including the hidden test rerun.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

input_root = Path(os.environ.get("NTL_INPUT_ROOT", "/kaggle/input"))
found = sorted(input_root.rglob("ntl_bundle.json"))
if not found:
    archives = sorted(input_root.rglob("cpu-kaggle-bundle-v1.zip"))
    if len(archives) != 1:
        raise RuntimeError("Add your private model-bundle dataset using Add Input; expected one bundle manifest or one cpu-kaggle-bundle-v1.zip")
    extracted = Path(tempfile.mkdtemp(prefix="ntl_bundle_"))
    with zipfile.ZipFile(archives[0]) as archive:
        members = archive.infolist()
        if sum(item.file_size for item in members) > 1024**3:
            raise RuntimeError("Unexpected model archive size")
        for item in members:
            target = (extracted / item.filename).resolve()
            if not target.is_relative_to(extracted.resolve()):
                raise RuntimeError("Unsafe archive path")
        archive.extractall(extracted)
    found = sorted(extracted.rglob("ntl_bundle.json"))
if len(found) != 1:
    raise RuntimeError(f"Attach exactly one model bundle; found {len(found)} manifests")
bundle = found[0].parent
manifest = json.loads(found[0].read_text())
minor = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Runtime: Python {minor}; bundle requires Python {manifest['python_minor']}")
if manifest["python_minor"] != minor:
    raise RuntimeError(f"Bundle Python {manifest['python_minor']} != runtime {minor}; report this message so wheels can be rebuilt")
for relative, expected in manifest["files"].items():
    file = (bundle / relative).resolve()
    if not file.is_relative_to(bundle.resolve()):
        raise RuntimeError("Unsafe bundle path")
    digest = hashlib.sha256()
    with file.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != expected:
        raise RuntimeError(f"Checksum mismatch: {relative}")
work = Path(os.environ.get("NTL_WORKING", "/kaggle/working"))
work.mkdir(parents=True, exist_ok=True)
# Fresh temporary directories make Run All repeatable and keep installed packages out of notebook outputs.
site = Path(tempfile.mkdtemp(prefix="ntl_site_"))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                "--find-links", str(bundle / "wheels"), "--only-binary=:all:",
                "--no-compile", "--target", str(site), manifest["package"]], check=True)
print("Model checksums verified; offline packages installed.")


In [ ]:
import uuid

test_path = Path(os.environ.get("NTL_TEST_PATH", "/kaggle/input/competitions/name-that-llm/test.csv"))
if not test_path.is_file():
    fallback = Path("/kaggle/input/name-that-llm/test.csv")
    if "NTL_TEST_PATH" not in os.environ and fallback.is_file():
        test_path = fallback
    else:
        raise RuntimeError(f"Competition test.csv missing at {test_path}; attach Name That LLM using Add Input")
output_path = work / "submission.parquet"
pending_output = work / f".submission-{uuid.uuid4().hex}.parquet"
env = {**os.environ, "PYTHONPATH": str(site), "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"}
# Import the bundled package versions in a fresh process; all inference is offline.
subprocess.run([sys.executable, "-m", "ntl.cli", "predict",
                "--model", str(bundle / "model.joblib"), "--test", str(test_path),
                "--out", str(pending_output), "--threads", "2",
                "--id-dtype", os.environ.get("NTL_ID_DTYPE", "string")], env=env, check=True)
# Replace the previous output only after the new prediction file has passed validation.
pending_output.replace(output_path)
print(f"SUCCESS: validated submission saved to {output_path}")


In [ ]:
preview = "import pandas as pd,sys; frame=pd.read_parquet(sys.argv[1]); print('Submission rows:',len(frame)); print(frame.head(10).to_string(index=False))"
subprocess.run([sys.executable, "-c", preview, str(output_path)], env=env, check=True)
